In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

file_path = (r"C:\Users\amit6\OneDrive\Desktop\Intelligent-Retail-Customer-Analytics\Data\Processed_data\retail_clean_master.csv")

df = pd.read_csv(
    file_path,
    parse_dates=["InvoiceDate"]
)

df.head()

C:\Users\amit6\AppData\Local\Temp\ipykernel_18140\4208956323.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


In [2]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.info()

Rows: 1007913
Columns: 8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007913 entries, 0 to 1007912
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007913 non-null  object        
 1   StockCode    1007913 non-null  object        
 2   Description  1007913 non-null  object        
 3   Quantity     1007913 non-null  int64         
 4   InvoiceDate  1007913 non-null  datetime64[ns]
 5   Price        1007913 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007913 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 61.5+ MB


# Transaction Features

In [3]:
df["TotalAmount"] = df["Quantity"] * df["Price"]

In [4]:
df[['Quantity','Price','TotalAmount']].head()

,Quantity,Price,TotalAmount
0,12,6.95,83.40
1,12,6.75,81.00
2,12,6.75,81.00
3,48,2.10,100.80
4,24,1.25,30.00


# Time Features

In [5]:
df["Year"] = df["InvoiceDate"].dt.year

df["Month"] = df["InvoiceDate"].dt.month

df["MonthName"] = df["InvoiceDate"].dt.month_name()

df["Quarter"] = df["InvoiceDate"].dt.quarter

df["Day"] = df["InvoiceDate"].dt.day

df["DayName"] = df["InvoiceDate"].dt.day_name()

df["Hour"] = df["InvoiceDate"].dt.hour

In [6]:

df[['Day','MonthName','Hour']]

,Day,MonthName,Hour
0,1,December,7
1,1,December,7
2,1,December,7
3,1,December,7
4,1,December,7
...,...,...,...
1007908,9,December,12
1007909,9,December,12
1007910,9,December,12
1007911,9,December,12


In [7]:
df["IsWeekend"] = (
    df["DayName"]
    .isin(["Saturday", "Sunday"])
)

df["IsWeekend"].head()

0    False
1    False
2    False
3    False
4    False
Name: IsWeekend, dtype: bool

# Invoice Features

In [8]:
## Net InvoiceTotal over Invoice
invoice_total = (
    df.groupby("Invoice")["TotalAmount"]
      .sum()
      .rename("InvoiceTotal")
)

df = df.merge(
    invoice_total,
    on="Invoice"
)

In [9]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount,Year,Month,MonthName,Quarter,Day,DayName,Hour,IsWeekend,InvoiceTotal
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom,83.40,2009,12,December,4,1,Tuesday,7,False,505.30
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,12,December,4,1,Tuesday,7,False,505.30
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,12,December,4,1,Tuesday,7,False,505.30
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom,100.80,2009,12,December,4,1,Tuesday,7,False,505.30
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom,30.00,2009,12,December,4,1,Tuesday,7,False,505.30


In [10]:
## Number of products purchase per Invoice
basket_size = (
    df.groupby("Invoice")["StockCode"]
      .count()
      .rename("BasketSize")
)

df = df.merge(
    basket_size,
    on="Invoice"
)

# Customer Features

In [11]:
purchase_frequency = (
    df.groupby("Customer ID")["Invoice"]
      .nunique()
      .rename("PurchaseFrequency")
)

df = df.merge(
    purchase_frequency,
    on="Customer ID",
    how="left"
)

In [12]:
avg_order = (
    df.groupby("Customer ID")["InvoiceTotal"]
      .mean()
      .rename("AverageOrderValue")
)

df = df.merge(
    avg_order,
    on="Customer ID",
    how="left"
)

In [13]:
customer_revenue = (
    df.groupby("Customer ID")["TotalAmount"]
      .sum()
      .rename("CustomerRevenue")
)

df = df.merge(
    customer_revenue,
    on="Customer ID",
    how="left"
)

# Product Features

In [14]:
product_count = (
    df.groupby("StockCode")["Invoice"]
      .count()
      .rename("ProductPurchaseCount")
)

df = df.merge(
    product_count,
    on="StockCode"
)

In [15]:
product_revenue = (
    df.groupby("StockCode")["TotalAmount"]
      .sum()
      .rename("ProductRevenue")
)

df = df.merge(
    product_revenue,
    on="StockCode"
)

In [16]:
df["CustomerType"] = pd.cut(
    df["PurchaseFrequency"],
    bins=[0,2,5,10,1000],
    labels=[
        "New",
        "Occasional",
        "Regular",
        "Loyal"
    ]
)

In [17]:
new_features = [
    "TotalAmount",
    "Year",
    "Month",
    "MonthName",
    "Quarter",
    "Day",
    "DayName",
    "Hour",
    "IsWeekend",
    "InvoiceTotal",
    "BasketSize",
    "PurchaseFrequency",
    "AverageOrderValue",
    "CustomerRevenue",
    "ProductPurchaseCount",
    "ProductRevenue",
    "CustomerType"
]

df[new_features].head()

,TotalAmount,Year,Month,MonthName,Quarter,Day,DayName,Hour,IsWeekend,InvoiceTotal,BasketSize,PurchaseFrequency,AverageOrderValue,CustomerRevenue,ProductPurchaseCount,ProductRevenue,CustomerType
0,83.40,2009,12,December,4,1,Tuesday,7,False,505.30,8,8.00,350.90,2433.28,522,21201.13,Regular
1,81.00,2009,12,December,4,1,Tuesday,7,False,505.30,8,8.00,350.90,2433.28,266,13787.93,Regular
2,81.00,2009,12,December,4,1,Tuesday,7,False,505.30,8,8.00,350.90,2433.28,350,18048.72,Regular
3,100.80,2009,12,December,4,1,Tuesday,7,False,505.30,8,8.00,350.90,2433.28,560,18650.87,Regular
4,30.00,2009,12,December,4,1,Tuesday,7,False,505.30,8,8.00,350.90,2433.28,2478,49300.57,Regular


In [18]:
df.to_csv(
    "retail_feature_engineered.csv",
    index=False
)

# Feature Engineering

## Objective

This notebook creates business-oriented features from the cleaned retail transaction data. These engineered features will be reused across SQL analysis, exploratory data analysis, Power BI dashboards, customer segmentation, RFM analysis, customer lifetime value estimation, market basket analysis, recommendation systems, and forecasting models.

The goal is to transform raw transactional records into analytically meaningful variables while maintaining a reproducible feature engineering pipeline.